# Wind Turbine ETL

**Purpose:** turn one month of raw hourly readings from a 15-turbine wind farm into clean, validated data and a daily power summary that flags unusual turbines.

**Input:** three CSV files (`data_group_1.csv`, `data_group_2.csv`, `data_group_3.csv`), each covering 5 turbines, with columns `timestamp`, `turbine_id`, `wind_speed`, `wind_direction` and `power_output` (MW). Period: 1–31 March 2022.

**Steps:**
1. Load and combine the three files
2. Check data quality (missing, duplicate, malformed and impossible values)
3. Convert to proper data types
4. Summarise daily minimum, maximum and average power for each turbine
5. Flag turbines whose daily average is more than 2 standard deviations from the fleet mean
6. Save the outputs and check them

**Outputs:**
- `cleaned_turbine_readings`: cleaned hourly data (11,160 rows)
- `daily_turbine_summary`: daily summary with fleet statistics and anomaly flag (465 rows, 20 flagged)

See `README.md` for how to run the notebook, assumptions and findings.

In [0]:
# Define the location for our raw CSV files

volume_path = "/Volumes/workspace/colibri_assessment/wind_turbine_data/"

# List the files stored in the volume

files = dbutils.fs.ls(volume_path)

for file in files:
    print(file.name)

cleaned_turbine_readings/
daily_turbine_summary/
data_group_1.csv
data_group_2.csv
data_group_3.csv


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType


## Data Ingestion & Schema Definition

**Objective:** Load three wind turbine CSV files into PySpark for data quality assessment and processing.

**Approach:**
- Define an explicit schema before ingestion.
- Initially read all columns as `StringType` to preserve raw sensor values.
- Inspect missing, malformed, duplicate and physically invalid readings.
- Convert validated values to appropriate types (`TimestampType`, `DoubleType`).
- Combine datasets using `unionByName()`.
- Document data quality findings before cleaning.

**Reason:** Preserve original values, prevent unnoticed parsing issues and ensure traceable cleaning decisions.

In [0]:
raw_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("turbine_id", StringType(), True),
    StructField("wind_speed", StringType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("power_output", StringType(), True)
])

In [0]:
# Load the first CSV file using the predefined schema
df1 = (
    spark.read
    .schema(raw_schema)
    .option("header", True)
    .csv(volume_path + "data_group_1.csv")
)

# Preview the first five records
df1.show(5, truncate=False)

+-------------------+----------+----------+--------------+------------+
|timestamp          |turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-01 00:00:00|1         |11.8      |169           |2.7         |
|2022-03-01 00:00:00|2         |11.6      |24            |2.2         |
|2022-03-01 00:00:00|3         |13.8      |335           |2.3         |
|2022-03-01 00:00:00|4         |12.8      |238           |1.9         |
|2022-03-01 00:00:00|5         |11.4      |103           |3.5         |
+-------------------+----------+----------+--------------+------------+
only showing top 5 rows


In [0]:
# Load the remaining two CSV files
df2 = (
    spark.read
    .schema(raw_schema)
    .option("header", True)
    .csv(volume_path + "data_group_2.csv")
)

df3 = (
    spark.read
    .schema(raw_schema)
    .option("header", True)
    .csv(volume_path + "data_group_3.csv")
)

# Check the number of records in each dataset
print("Group 1:", df1.count())
print("Group 2:", df2.count())
print("Group 3:", df3.count())

Group 1: 3720
Group 2: 3720
Group 3: 3720


In [0]:
# Combine all three datasets
raw_df = df1.unionByName(df2).unionByName(df3)
raw_count = raw_df.count()

# Verify the combined dataset
print("Total raw records:", raw_count)

# Preview the combined data
raw_df.show(5, truncate=False)

Total raw records: 11160
+-------------------+----------+----------+--------------+------------+
|timestamp          |turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-01 00:00:00|1         |11.8      |169           |2.7         |
|2022-03-01 00:00:00|2         |11.6      |24            |2.2         |
|2022-03-01 00:00:00|3         |13.8      |335           |2.3         |
|2022-03-01 00:00:00|4         |12.8      |238           |1.9         |
|2022-03-01 00:00:00|5         |11.4      |103           |3.5         |
+-------------------+----------+----------+--------------+------------+
only showing top 5 rows


In [0]:
# Check the structure and data types
raw_df.printSchema()

# Count distinct turbines
print("Unique turbines:", raw_df.select("turbine_id").distinct().count())

# Check the timestamp range
raw_df.select(
    F.min("timestamp").alias("earliest_timestamp"),
    F.max("timestamp").alias("latest_timestamp")
).show()

# Display turbine IDs
raw_df.select("turbine_id").distinct().orderBy("turbine_id").show(20)

root
 |-- timestamp: string (nullable = true)
 |-- turbine_id: string (nullable = true)
 |-- wind_speed: string (nullable = true)
 |-- wind_direction: string (nullable = true)
 |-- power_output: string (nullable = true)

Unique turbines: 15
+-------------------+-------------------+
| earliest_timestamp|   latest_timestamp|
+-------------------+-------------------+
|2022-03-01 00:00:00|2022-03-31 23:00:00|
+-------------------+-------------------+

+----------+
|turbine_id|
+----------+
|         1|
|        10|
|        11|
|        12|
|        13|
|        14|
|        15|
|         2|
|         3|
|         4|
|         5|
|         6|
|         7|
|         8|
|         9|
+----------+



### Missing Value Analysis
Check each column for null, empty or whitespace-only values before cleaning.

In [0]:
  
# Check missing values in each column
for column in raw_df.columns:
    missing = raw_df.filter(
        col(column).isNull() | (F.trim(col(column)) == "")
    ).count()

    print(column, ":", missing)

timestamp : 0
turbine_id : 0
wind_speed : 0
wind_direction : 0
power_output : 0


### Duplicate Record Analysis
Identify exact duplicate rows before cleaning, while preserving the original dataset.

In [0]:
# Count exact duplicate records

unique_rows = raw_df.dropDuplicates().count()

duplicate_rows = raw_count - unique_rows

print("Total records:", raw_count)
print("Unique records:", unique_rows)
print("Exact duplicates:", duplicate_rows)

Total records: 11160
Unique records: 11160
Exact duplicates: 0


### Duplicate Turbine/Timestamp Check
Identify multiple readings recorded for the same turbine at the same timestamp.

In [0]:
# Find repeated turbine/timestamp combinations
duplicate_keys = (
    raw_df
    .groupBy("turbine_id", "timestamp")
    .count()
    .filter(col("count") > 1)
)

# Display duplicate combinations
duplicate_keys.show(20, truncate=False)

# Count affected combinations
print("Duplicate turbine/timestamp combinations:", duplicate_keys.count())

+----------+---------+-----+
|turbine_id|timestamp|count|
+----------+---------+-----+
+----------+---------+-----+

Duplicate turbine/timestamp combinations: 0


### Numeric Data Type Validation
Check whether sensor readings can be converted to numeric values without losing invalid original data.

In [0]:
# Check for values that cannot be converted to numbers
for column in ["wind_speed", "wind_direction", "power_output"]:
    invalid = raw_df.filter(
        col(column).isNotNull() &
        col(column).try_cast("double").isNull()
    ).count()

    print(column, "invalid numeric values:", invalid)

wind_speed invalid numeric values: 0
wind_direction invalid numeric values: 0
power_output invalid numeric values: 0


### Sensor Range Analysis
Inspect minimum and maximum sensor readings to identify potentially impossible values.

In [0]:
# Convert sensor readings to numeric values
profile_df = (
    raw_df
    .withColumn("wind_speed", col("wind_speed").cast("double"))
    .withColumn("wind_direction", col("wind_direction").cast("double"))
    .withColumn("power_output", col("power_output").cast("double"))
)

# Check minimum and maximum readings
profile_df.select(
    F.min("wind_speed").alias("min_wind_speed"),
    F.max("wind_speed").alias("max_wind_speed"),
    F.min("wind_direction").alias("min_wind_direction"),
    F.max("wind_direction").alias("max_wind_direction"),
    F.min("power_output").alias("min_power_output"),
    F.max("power_output").alias("max_power_output")
).show()

# Assumed physical limits: wind speed >= 0 m/s, wind direction between 0-360 degrees, power output >= 0 MW

invalid_readings = profile_df.filter(
    (col("wind_speed") < 0) | 
    (col("wind_direction") < 0) | (col("wind_direction") > 360) |
    (col("power_output") < 0)
).count()

print("Physically impossible readings:", invalid_readings)

+--------------+--------------+------------------+------------------+----------------+----------------+
|min_wind_speed|max_wind_speed|min_wind_direction|max_wind_direction|min_power_output|max_power_output|
+--------------+--------------+------------------+------------------+----------------+----------------+
|           9.0|          15.0|               0.0|             359.0|             1.5|             4.5|
+--------------+--------------+------------------+------------------+----------------+----------------+

Physically impossible readings: 0


### Timestamp Validation
Verify that all raw timestamps can be converted to valid datetime values.

In [0]:
# Check timestamps that cannot be parsed
invalid_timestamps = raw_df.filter(
    F.try_to_timestamp(
        col("timestamp"),
        F.lit("yyyy-MM-dd HH:mm:ss")
    ).isNull()
).count()

print("Invalid timestamps:", invalid_timestamps)

Invalid timestamps: 0


### Turbine Reading Completeness
Verify that each of the 15 turbines has the expected 744 hourly readings for March 2022.

In [0]:
# Count readings for each turbine
turbine_counts = (
    raw_df
    .groupBy("turbine_id")
    .count()
    .orderBy(col("turbine_id").cast("int"))
)

turbine_counts.show(15)

+----------+-----+
|turbine_id|count|
+----------+-----+
|         1|  744|
|         2|  744|
|         3|  744|
|         4|  744|
|         5|  744|
|         6|  744|
|         7|  744|
|         8|  744|
|         9|  744|
|        10|  744|
|        11|  744|
|        12|  744|
|        13|  744|
|        14|  744|
|        15|  744|
+----------+-----+



### Hourly Timestamp Coverage
Check that readings occur on exact hourly intervals and that March contains all 744 expected timestamps.

In [0]:
# Convert timestamp strings to datetime values
time_df = raw_df.withColumn(
    "parsed_time",
    F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss")
)

# Count readings not recorded on the hour
invalid_hours = time_df.filter(
    (F.minute("parsed_time") != 0) |
    (F.second("parsed_time") != 0)
).count()

# Count distinct timestamps
unique_hours = time_df.select("parsed_time").distinct().count()

print("Readings outside hourly intervals:", invalid_hours)
print("Distinct timestamps:", unique_hours)

Readings outside hourly intervals: 0
Distinct timestamps: 744


### Data Profiling Summary

| Check | Result |
|---|---|
| Records | 11,160 (15 turbines × 744 hourly readings), 1–31 March 2022 |
| Missing or blank values | 0 |
| Duplicate rows / duplicate turbine-timestamp pairs | 0 / 0 |
| Non-numeric values / unparseable timestamps | 0 / 0 |
| Readings not on the hour | 0 |
| Physically impossible readings (speed < 0, direction outside 0–360°, power < 0) | 0 |
| Observed ranges | Wind speed 9–15, direction 0–359°, power 1.5–4.5 MW |

**Conclusion:** no data quality issues found, so no records were removed or changed. Upper limits (such as rated power) were not available, so only lower and directional limits were checked.

### Data Type Conversion
Convert validated raw readings into appropriate data types for processing and daily aggregation.

In [0]:
# Convert validated readings to appropriate data types
clean_df = (
    raw_df
    .withColumn("timestamp", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("turbine_id", col("turbine_id").cast("int"))
    .withColumn("wind_speed", col("wind_speed").cast("double"))
    .withColumn("wind_direction", col("wind_direction").cast("double"))
    .withColumn("power_output", col("power_output").cast("double"))
)

# Verify the new data types
clean_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- turbine_id: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- power_output: double (nullable = true)



### Validate Converted Data
Verify record counts and check for null values after data type conversion.

In [0]:
# Check total records after conversion
print("Raw records:", raw_count)

clean_count = clean_df.count()
print("Clean records:", clean_count)

# Check null values in converted columns
for column in clean_df.columns:
    null_count = clean_df.filter(
        col(column).isNull()
    ).count()

    print(column, "nulls:", null_count)

Raw records: 11160
Clean records: 11160
timestamp nulls: 0
turbine_id nulls: 0
wind_speed nulls: 0
wind_direction nulls: 0
power_output nulls: 0


### Extract Date from Timestamp

Extract the date from each hourly timestamp using PySpark's `to_date()` function.

This prepares the cleaned dataset for daily aggregation, allowing us to calculate the minimum, maximum and average power output for each turbine per day.

In [0]:
# Extract date from timestamp
daily_df = clean_df.withColumn(
    "date",
    F.to_date("timestamp")
)

# Preview the result
daily_df.select("timestamp", "date", "turbine_id", "power_output").show(5)

+-------------------+----------+----------+------------+
|          timestamp|      date|turbine_id|power_output|
+-------------------+----------+----------+------------+
|2022-03-01 00:00:00|2022-03-01|         1|         2.7|
|2022-03-01 00:00:00|2022-03-01|         2|         2.2|
|2022-03-01 00:00:00|2022-03-01|         3|         2.3|
|2022-03-01 00:00:00|2022-03-01|         4|         1.9|
|2022-03-01 00:00:00|2022-03-01|         5|         3.5|
+-------------------+----------+----------+------------+
only showing top 5 rows


### Daily Power Aggregation
Calculate minimum, maximum and average daily power output (MW) for each turbine.

In [0]:
# Calculate daily power statistics for each turbine
daily_summary = (
    daily_df
    .groupBy("turbine_id", "date")
    .agg(
        F.min("power_output").alias("min_power"),
        F.max("power_output").alias("max_power"),
        F.avg("power_output").alias("avg_power")
    )
    .orderBy("date", "turbine_id")
)

# Preview daily results
daily_summary.show(15)

+----------+----------+---------+---------+------------------+
|turbine_id|      date|min_power|max_power|         avg_power|
+----------+----------+---------+---------+------------------+
|         1|2022-03-01|      1.6|      4.4|2.9749999999999996|
|         2|2022-03-01|      1.6|      4.4|2.9833333333333325|
|         3|2022-03-01|      1.6|      4.5| 2.766666666666667|
|         4|2022-03-01|      1.5|      4.4| 3.020833333333334|
|         5|2022-03-01|      1.7|      4.5| 3.079166666666667|
|         6|2022-03-01|      1.5|      4.5| 2.908333333333333|
|         7|2022-03-01|      1.6|      4.5| 3.087499999999999|
|         8|2022-03-01|      1.6|      4.4| 3.016666666666666|
|         9|2022-03-01|      1.5|      4.3|2.7208333333333328|
|        10|2022-03-01|      1.5|      4.5|3.1250000000000004|
|        11|2022-03-01|      1.5|      4.4|2.7375000000000003|
|        12|2022-03-01|      1.6|      4.5|3.1583333333333337|
|        13|2022-03-01|      1.8|      4.3|3.0333333333

### Daily Aggregation Validation
Verify that all 15 turbines have a daily summary for each of the 31 days in March.

In [0]:
# Verify the number of daily summaries
print("Daily summary records:", daily_summary.count())

# Check the number of days per turbine
daily_summary.groupBy("turbine_id").count().orderBy("turbine_id").show(15)

Daily summary records: 465
+----------+-----+
|turbine_id|count|
+----------+-----+
|         1|   31|
|         2|   31|
|         3|   31|
|         4|   31|
|         5|   31|
|         6|   31|
|         7|   31|
|         8|   31|
|         9|   31|
|        10|   31|
|        11|   31|
|        12|   31|
|        13|   31|
|        14|   31|
|        15|   31|
+----------+-----+



### Fleet Daily Statistics
Calculate the daily fleet mean and population standard deviation of turbine average power outputs.

In [0]:
# Calculate fleet statistics for each day
fleet_stats = (
    daily_summary
    .groupBy("date")
    .agg(
        F.avg("avg_power").alias("fleet_mean"),
        F.stddev_pop("avg_power").alias("fleet_stddev")
    )
    .orderBy("date")
)

# Preview the first five days
fleet_stats.show(5)

+----------+------------------+-------------------+
|      date|        fleet_mean|       fleet_stddev|
+----------+------------------+-------------------+
|2022-03-01| 2.997777777777777|0.16426651409925583|
|2022-03-02| 3.023611111111111|0.21708648785651438|
|2022-03-03|3.0272222222222216|0.14619294286390797|
|2022-03-04|2.9647222222222216|0.16015665556256992|
|2022-03-05|2.9372222222222226|0.21304921763715162|
+----------+------------------+-------------------+
only showing top 5 rows


### Turbine Anomaly Detection
Compare each turbine's daily average power with the fleet mean and flag values more than two standard deviations away.


In [0]:
# Join daily turbine results with fleet statistics
anomaly_df = daily_summary.join(
    fleet_stats,
    on="date",
    how="left"
)

# Flag unusual turbine performance
anomaly_df = anomaly_df.withColumn(
    "is_anomaly",
    F.when(
        F.abs(col("avg_power") - col("fleet_mean")) >
        2 * col("fleet_stddev"),
        True
    ).otherwise(False)
)

# Display flagged turbines
anomaly_df.filter(col("is_anomaly") == True).show(20, truncate=False)

+----------+----------+---------+---------+------------------+------------------+-------------------+----------+
|date      |turbine_id|min_power|max_power|avg_power         |fleet_mean        |fleet_stddev       |is_anomaly|
+----------+----------+---------+---------+------------------+------------------+-------------------+----------+
|2022-03-25|2         |1.6      |4.3      |2.7249999999999996|3.065             |0.16870146764383892|true      |
|2022-03-20|4         |1.6      |4.2      |2.5041666666666673|2.976666666666667 |0.22648072438624522|true      |
|2022-03-26|4         |1.5      |4.4      |2.4708333333333337|2.9777777777777783|0.20791996026486545|true      |
|2022-03-18|2         |1.5      |4.3      |2.7375000000000003|3.0047222222222225|0.1296707606703975 |true      |
|2022-03-07|1         |1.9      |4.3      |3.283333333333333 |2.995833333333333 |0.13825968588399423|true      |
|2022-03-11|4         |1.5      |4.3      |2.479166666666666 |2.9741666666666666|0.2003191897392

### Anomaly Summary
Count all flagged turbine-days and display their details for investigation.

In [0]:
# Filter anomalous turbine-days
flagged_df = anomaly_df.filter(col("is_anomaly") == True)

# Count all flagged records
flagged_count = flagged_df.count()
print("Total flagged turbine-days:", flagged_count)

# Display flagged records in chronological order
display(flagged_df.orderBy("date", "turbine_id"))

Total flagged turbine-days: 20


date,turbine_id,min_power,max_power,avg_power,fleet_mean,fleet_stddev,is_anomaly
2022-03-01,15,2.0,4.5,3.3708333333333336,2.997777777777777,0.16426651409925583,true
2022-03-02,2,1.6,4.2,2.4666666666666663,3.023611111111111,0.21708648785651438,true
2022-03-03,12,1.7,4.2,2.670833333333333,3.0272222222222216,0.14619294286390797,true
2022-03-04,8,1.5,4.1,2.5291666666666663,2.9647222222222216,0.16015665556256992,true
2022-03-06,3,1.7,4.4,3.3000000000000007,3.0066666666666673,0.13078997866242892,true
2022-03-07,1,1.9,4.3,3.283333333333333,2.995833333333333,0.13825968588399423,true
2022-03-08,11,1.7,4.3,2.579166666666667,3.0730555555555563,0.21425941931553918,true
2022-03-11,4,1.5,4.3,2.479166666666666,2.9741666666666666,0.20031918973922044,true
2022-03-12,12,1.6,4.2,2.6375,2.9858333333333333,0.13952465599179806,true
2022-03-13,7,1.6,4.5,2.7249999999999996,2.9844444444444442,0.11896998791582425,true


### Final Output Validation
Verify cleaned hourly records, daily summaries and anomaly counts before saving.

In [0]:
# Validate final output record counts
print("Cleaned hourly records:", clean_count)
print("Daily summary records:", anomaly_df.count())
print("Flagged turbine-days:", flagged_count)

Cleaned hourly records: 11160
Daily summary records: 465
Flagged turbine-days: 20


### Save Cleaned Hourly Readings

Export the validated hourly turbine readings to CSV format for easy access, review and downstream analysis.

In [0]:
# Save cleaned hourly readings as CSV
clean_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(volume_path + "cleaned_turbine_readings")

print("Cleaned hourly readings saved successfully.")

Cleaned hourly readings saved successfully.


### Save Daily Turbine Summary

Export the daily minimum, maximum and average power output for each turbine, including fleet statistics and anomaly flags.

The output contains 465 turbine-day records, including 20 flagged anomalies.

In [0]:
# Save daily summary with anomaly flags as CSV
anomaly_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(volume_path + "daily_turbine_summary")

print("Daily turbine summary saved successfully.")

Daily turbine summary saved successfully.


### Verify Saved Outputs

Read both exported CSV datasets and validate their record counts against the original processed DataFrames.

In [0]:
# Read saved CSV outputs
saved_clean_df = spark.read.option("header", True).csv(
    volume_path + "cleaned_turbine_readings"
)

saved_summary_df = spark.read.option("header", True).csv(
    volume_path + "daily_turbine_summary"
)

# Verify record counts
print("Saved cleaned records:", saved_clean_df.count())
print("Saved daily summaries:", saved_summary_df.count())

Saved cleaned records: 11160
Saved daily summaries: 465


### Verify Exported Anomaly Flags

Confirm that the saved daily summary retains all 20 anomalous turbine-day records.

In [0]:
# Count anomalies in the saved CSV
saved_anomalies = saved_summary_df.filter(
    col("is_anomaly") == "true"
)

print("Saved flagged turbine-days:", saved_anomalies.count())

Saved flagged turbine-days: 20


### Summary

- **Loaded:** 3 CSV files, 11,160 hourly readings, 15 turbines, March 2022. All rows kept.
- **Summarised:** daily min, max and average power per turbine (465 turbine-days).
- **Anomalies:** a turbine-day is flagged when its average is more than 2 population standard deviations from that day's fleet mean. 20 of 465 (4.3%) were flagged and kept for investigation.
- **Saved:** cleaned hourly data (11,160 rows) and daily summary (465 rows) as CSV. Read-back counts match.